# Petri_curcle: split, train (`yolo26n.pt` + `yolo26s.pt`), test, crop 736x736


In [ ]:
# Uncomment if required
# %pip install -q ultralytics pyyaml pandas matplotlib pillow opencv-python

from pathlib import Path
import random
import shutil
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import yaml
from ultralytics import YOLO
from IPython.display import display
import torch

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "Petri_curcle"

SOURCE_IMAGES = DATASET_ROOT / "images" / "train"
SOURCE_LABELS = DATASET_ROOT / "labels" / "train"

SPLIT_ROOT = DATASET_ROOT / "split_dataset"
SPLITS = {"train": 0.7, "val": 0.2, "test": 0.1}
assert abs(sum(SPLITS.values()) - 1.0) < 1e-9

MODEL_CONFIGS = {
    "n": {"weights": "yolo26n.pt", "train_name": "yolo26n_petri_curcle"},
    "s": {"weights": "yolo26s.pt", "train_name": "yolo26s_petri_curcle"},
}

EPOCHS = 100
IMGSZ = 736
BATCH = 8
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

TRAIN_PROJECT = "runs/petri_curcle"
PREDICT_MODEL_KEY = "s"  # which trained model to use for preview + crop

# Offline soft augmentation (files are physically added to split_dataset)
USE_OFFLINE_SOFT_AUG = True
AUG_TARGET_SPLITS = ["train"]
AUG_MULTIPLIER = 5  # total factor including originals
AUG_CLEAR_PREVIOUS = True

# Online soft augmentation (Ultralytics train args)
YOLO_TRAIN_AUG_ARGS = {
    "hsv_h": 0.01,
    "hsv_s": 0.22,
    "hsv_v": 0.14,
    "degrees": 3.0,
    "translate": 0.04,
    "scale": 0.08,
    "shear": 1.0,
    "perspective": 0.0,
    "fliplr": 0.5,
    "flipud": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "erasing": 0.1,
}

# Crop settings for predict workflow
TARGET_W = 736
TARGET_H = 736
CROP_CONF = 0.25
CROP_IOU = 0.5
OVERWRITE_CROPS = True

print(f"Dataset root: {DATASET_ROOT.resolve()}")
print(f"Source images: {SOURCE_IMAGES}")
print(f"Source labels: {SOURCE_LABELS}")
print(f"Model configs: {MODEL_CONFIGS}")
print(f"Device: {DEVICE}")
print(f"Offline aug enabled: {USE_OFFLINE_SOFT_AUG}, multiplier={AUG_MULTIPLIER}, splits={AUG_TARGET_SPLITS}")


In [ ]:
# Split source train set into train/val/test and create data_split.yaml
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
image_files = sorted([p for p in SOURCE_IMAGES.glob("*") if p.suffix.lower() in image_exts])

pairs = []
missing_labels = []
for img in image_files:
    lbl = SOURCE_LABELS / f"{img.stem}.txt"
    if lbl.exists():
        pairs.append((img, lbl))
    else:
        missing_labels.append(img.name)

if not pairs:
    raise RuntimeError("No image-label pairs found in source directories.")

idx = list(range(len(pairs)))
rng = random.Random(SEED)
rng.shuffle(idx)

n_total = len(idx)
n_train = int(n_total * SPLITS["train"])
n_val = int(n_total * SPLITS["val"])
n_test = n_total - n_train - n_val

split_indices = {
    "train": idx[:n_train],
    "val": idx[n_train:n_train + n_val],
    "test": idx[n_train + n_val:],
}

if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)

for split in split_indices:
    (SPLIT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (SPLIT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

for split, indices in split_indices.items():
    for i in indices:
        img, lbl = pairs[i]
        shutil.copy2(img, SPLIT_ROOT / "images" / split / img.name)
        shutil.copy2(lbl, SPLIT_ROOT / "labels" / split / lbl.name)

data_yaml = {
    "path": str(SPLIT_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "Petri_curcle"},
}

data_yaml_path = SPLIT_ROOT / "data_split.yaml"
with data_yaml_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

summary_df = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "images": [len(split_indices["train"]), len(split_indices["val"]), len(split_indices["test"])],
    }
)
summary_df["labels"] = summary_df["images"]

if missing_labels:
    print(f"Missing labels: {len(missing_labels)} files (ignored)")

print(f"Total valid pairs: {len(pairs)}")
print(f"Split root: {SPLIT_ROOT.resolve()}")
print(f"data yaml: {data_yaml_path}")
display(summary_df)


In [ ]:
# Soft offline augmentation for Petri_curcle/split_dataset
# Creates extra files like IMG_1234__soft01.jpg + IMG_1234__soft01.txt
import re

soft_suffix_re = re.compile(r"__soft\d{2}$")
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def gamma_correct(image, gamma):
    lut = np.array([((x / 255.0) ** gamma) * 255.0 for x in range(256)], dtype=np.float32)
    lut = np.clip(lut, 0, 255).astype(np.uint8)
    return cv2.LUT(image, lut)


def mild_color_jitter(image, rng):
    out = image.astype(np.float32)
    alpha = float(rng.uniform(0.95, 1.07))
    beta = float(rng.uniform(-8.0, 8.0))
    out = np.clip(out * alpha + beta, 0, 255).astype(np.uint8)
    out = gamma_correct(out, gamma=float(rng.uniform(0.93, 1.07)))

    hsv = cv2.cvtColor(out, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[..., 1] *= float(rng.uniform(0.93, 1.08))
    hsv[..., 2] *= float(rng.uniform(0.95, 1.05))
    return cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR)


def mild_noise_blur(image, rng):
    sigma_noise = float(rng.uniform(2.0, 5.0))
    noisy = image.astype(np.float32) + rng.normal(0.0, sigma_noise, size=image.shape).astype(np.float32)
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    k = 3 if float(rng.random()) < 0.75 else 5
    sigma_blur = float(rng.uniform(0.2, 0.9))
    return cv2.GaussianBlur(noisy, (k, k), sigmaX=sigma_blur)


def mild_clahe(image, rng):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=float(rng.uniform(1.2, 1.8)), tileGridSize=(8, 8))
    l2 = clahe.apply(l)
    out = cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)
    return mild_color_jitter(out, rng)


def mild_sharpen(image, rng):
    blur = cv2.GaussianBlur(image, (0, 0), sigmaX=float(rng.uniform(0.4, 0.8)))
    return cv2.addWeighted(image, 1.12, blur, -0.12, 0.0)


def soft_augment(image, variant_idx, rng):
    mode = (variant_idx - 1) % 4
    if mode == 0:
        return mild_color_jitter(image, rng)
    if mode == 1:
        return mild_noise_blur(image, rng)
    if mode == 2:
        return mild_clahe(image, rng)
    return mild_sharpen(mild_color_jitter(image, rng), rng)


def collect_pairs(images_dir, labels_dir):
    pairs = []
    for img_path in sorted(images_dir.glob("*")):
        if not img_path.is_file() or img_path.suffix.lower() not in image_exts:
            continue
        if soft_suffix_re.search(img_path.stem):
            continue
        lbl_path = labels_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            pairs.append((img_path, lbl_path))
    return pairs


def clear_previous_soft(images_dir, labels_dir):
    removed = 0
    for p in images_dir.glob("*"):
        if p.is_file() and soft_suffix_re.search(p.stem):
            p.unlink()
            removed += 1
    for p in labels_dir.glob("*.txt"):
        if p.is_file() and soft_suffix_re.search(p.stem):
            p.unlink()
            removed += 1
    return removed


aug_report = {
    "dataset_root": str(SPLIT_ROOT.resolve()),
    "splits": {},
    "settings": {
        "use_offline_soft_aug": bool(USE_OFFLINE_SOFT_AUG),
        "aug_target_splits": list(AUG_TARGET_SPLITS),
        "aug_multiplier": int(AUG_MULTIPLIER),
        "aug_clear_previous": bool(AUG_CLEAR_PREVIOUS),
    },
}

if not USE_OFFLINE_SOFT_AUG:
    print("Offline augmentation disabled, skipping.")
else:
    if AUG_MULTIPLIER < 2:
        raise ValueError("AUG_MULTIPLIER must be >= 2")

    rng = np.random.default_rng(SEED)
    for split in AUG_TARGET_SPLITS:
        images_dir = SPLIT_ROOT / "images" / split
        labels_dir = SPLIT_ROOT / "labels" / split
        if not images_dir.exists() or not labels_dir.exists():
            raise FileNotFoundError(f"Split not found: {split}")

        removed = clear_previous_soft(images_dir, labels_dir) if AUG_CLEAR_PREVIOUS else 0
        base_pairs = collect_pairs(images_dir, labels_dir)

        created = 0
        failed = 0
        for img_path, lbl_path in base_pairs:
            image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
            if image is None:
                failed += (AUG_MULTIPLIER - 1)
                continue

            label_text = lbl_path.read_text(encoding="utf-8")
            for aug_idx in range(1, AUG_MULTIPLIER):
                out_stem = f"{img_path.stem}__soft{aug_idx:02d}"
                out_img = images_dir / f"{out_stem}{img_path.suffix.lower()}"
                out_lbl = labels_dir / f"{out_stem}.txt"

                aug_img = soft_augment(image, aug_idx, rng)
                ok = cv2.imwrite(str(out_img), aug_img)
                if not ok:
                    failed += 1
                    continue
                out_lbl.write_text(label_text, encoding="utf-8")
                created += 1

        all_img_stems = {
            p.stem for p in images_dir.glob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        }
        all_lbl_stems = {p.stem for p in labels_dir.glob("*.txt") if p.is_file()}
        matched = len(all_img_stems & all_lbl_stems)

        aug_report["splits"][split] = {
            "base_pairs": len(base_pairs),
            "created_pairs": created,
            "failed_pairs": failed,
            "removed_previous_soft_files": removed,
            "matched_pairs_after": matched,
            "expected_after": len(base_pairs) * AUG_MULTIPLIER,
        }

        print(
            f"[{split}] base={len(base_pairs)} created={created} failed={failed} "
            f"matched_after={matched} expected={len(base_pairs) * AUG_MULTIPLIER}"
        )

    aug_report["generated_at_utc"] = pd.Timestamp.utcnow().isoformat()
    aug_report_path = SPLIT_ROOT / "soft_aug_report_notebook.json"
    with aug_report_path.open("w", encoding="utf-8") as f:
        json.dump(aug_report, f, ensure_ascii=False, indent=2)
    print(f"Saved augmentation report: {aug_report_path}")


In [ ]:
# Train YOLO for all configured models (n + s)
train_runs = {}

for model_key, cfg in MODEL_CONFIGS.items():
    print(f"\n=== Training model: {model_key} ({cfg['weights']}) ===")
    model = YOLO(cfg["weights"])

    _ = model.train(
        data='C:/ColonyNet/Petri_curcle/split_dataset/data_split.yaml',
        epochs=50,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=TRAIN_PROJECT,
        name=cfg["train_name"],
        exist_ok=True,
        plots=True,
        **YOLO_TRAIN_AUG_ARGS,
    )

    run_dir = Path(model.trainer.save_dir)
    best_weights = run_dir / "weights" / "best.pt"
    if not best_weights.exists():
        raise FileNotFoundError(f"Best checkpoint not found: {best_weights}")

    train_runs[model_key] = {
        "weights": cfg["weights"],
        "run_dir": run_dir,
        "best_weights": best_weights,
    }

train_runs_df = pd.DataFrame(
    [
        {
            "model": k,
            "base_weights": v["weights"],
            "run_dir": str(v["run_dir"]),
            "best_weights": str(v["best_weights"]),
        }
        for k, v in train_runs.items()
    ]
)
display(train_runs_df)


In [ ]:
# Evaluate both best checkpoints on test split

def to_float(v):
    try:
        return float(v)
    except Exception:
        return float("nan")


eval_rows = []
for model_key, info in train_runs.items():
    best_model = YOLO(str(info["best_weights"]))
    test_metrics = best_model.val(
        data=str(data_yaml_path),
        split="test",
        project=TRAIN_PROJECT,
        name=f"{MODEL_CONFIGS[model_key]['train_name']}_test",
        exist_ok=True,
        plots=True,
        save_json=True,
    )

    eval_rows.append(
        {
            "model": model_key,
            "precision_B": to_float(test_metrics.box.mp),
            "recall_B": to_float(test_metrics.box.mr),
            "mAP50_B": to_float(test_metrics.box.map50),
            "mAP50_95_B": to_float(test_metrics.box.map),
            "fitness": to_float(getattr(test_metrics, "fitness", float("nan"))),
            "test_dir": str(Path(test_metrics.save_dir)),
        }
    )

metrics_df = pd.DataFrame(eval_rows).sort_values("mAP50_95_B", ascending=False).reset_index(drop=True)
display(metrics_df)

metrics_csv_path = SPLIT_ROOT / "multi_model_test_metrics.csv"
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Saved metrics table: {metrics_csv_path}")

predict_model_key = PREDICT_MODEL_KEY if PREDICT_MODEL_KEY in train_runs else metrics_df.loc[0, "model"]
predict_weights = train_runs[predict_model_key]["best_weights"]
print(f"Predict/crop model key: {predict_model_key}")
print(f"Predict/crop weights: {predict_weights}")


In [ ]:
# Show sample predictions on test images for selected model
sample_images = sorted((SPLIT_ROOT / "images" / "test").glob("*"))
if not sample_images:
    print("No test images found for prediction preview.")
else:
    sample_images = sample_images[:6]
    pred_model = YOLO(str(predict_weights))
    preds = pred_model.predict(
        source=[str(p) for p in sample_images],
        conf=0.25,
        iou=0.5,
        imgsz=max(TARGET_W, TARGET_H),
        verbose=False,
    )

    n = len(sample_images)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, pred, src in zip(axes, preds, sample_images):
        annotated = pred.plot()  # BGR
        ax.imshow(annotated[..., ::-1])
        ax.set_title(src.name)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


## Crop Detected Petri Dishes and Resize to 736x736


In [ ]:
# Crop inference configuration
CROP_SOURCE_DIR = DATASET_ROOT / "images" / "train"
CROP_OUTPUT_DIR = DATASET_ROOT / f"cropped_{TARGET_W}"
CROP_REPORT_CSV = CROP_OUTPUT_DIR / "crop_report.csv"
CROP_MODEL_PATH = predict_weights

print(f"Crop model: {Path(CROP_MODEL_PATH)}")
print(f"Source dir: {CROP_SOURCE_DIR}")
print(f"Output dir: {CROP_OUTPUT_DIR}")
print(f"Target size: {TARGET_W}x{TARGET_H}")


In [ ]:
# Run detection-based cropping and resize to 736x736
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

if not CROP_SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {CROP_SOURCE_DIR}")
if not Path(CROP_MODEL_PATH).exists():
    raise FileNotFoundError(f"Model weights not found: {CROP_MODEL_PATH}")

if OVERWRITE_CROPS and CROP_OUTPUT_DIR.exists():
    shutil.rmtree(CROP_OUTPUT_DIR)
CROP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

crop_model = YOLO(str(CROP_MODEL_PATH))
source_images = sorted([p for p in CROP_SOURCE_DIR.glob("*") if p.suffix.lower() in image_exts])

if not source_images:
    raise RuntimeError(f"No images found in: {CROP_SOURCE_DIR}")

rows = []
for i, img_path in enumerate(source_images, start=1):
    img = cv2.imread(str(img_path))
    if img is None:
        rows.append({"file": img_path.name, "status": "read_error"})
        continue

    h, w = img.shape[:2]
    pred = crop_model.predict(
        source=str(img_path),
        conf=CROP_CONF,
        iou=CROP_IOU,
        imgsz=max(TARGET_W, TARGET_H),
        verbose=False,
    )[0]

    if pred.boxes is None or len(pred.boxes) == 0:
        rows.append({"file": img_path.name, "status": "no_detection", "orig_w": w, "orig_h": h})
        continue

    confs = pred.boxes.conf.detach().cpu().numpy()
    boxes = pred.boxes.xyxy.detach().cpu().numpy()
    best_idx = int(np.argmax(confs))

    x1, y1, x2, y2 = boxes[best_idx]
    x1 = max(0, int(np.floor(x1)))
    y1 = max(0, int(np.floor(y1)))
    x2 = min(w, int(np.ceil(x2)))
    y2 = min(h, int(np.ceil(y2)))

    if x2 <= x1 or y2 <= y1:
        rows.append({"file": img_path.name, "status": "invalid_bbox", "orig_w": w, "orig_h": h})
        continue

    crop = img[y1:y2, x1:x2]
    interp = cv2.INTER_AREA if crop.shape[1] >= TARGET_W and crop.shape[0] >= TARGET_H else cv2.INTER_LINEAR
    resized = cv2.resize(crop, (TARGET_W, TARGET_H), interpolation=interp)

    out_path = CROP_OUTPUT_DIR / img_path.name
    cv2.imwrite(str(out_path), resized)

    rows.append(
        {
            "file": img_path.name,
            "status": "saved",
            "conf": float(confs[best_idx]),
            "orig_w": w,
            "orig_h": h,
            "bbox_w": x2 - x1,
            "bbox_h": y2 - y1,
            "out_w": TARGET_W,
            "out_h": TARGET_H,
            "out_path": str(out_path),
        }
    )

    if i % 25 == 0 or i == len(source_images):
        print(f"Processed {i}/{len(source_images)}")

report_df = pd.DataFrame(rows)
status_counts = report_df["status"].value_counts().rename_axis("status").reset_index(name="count")
display(status_counts)

report_df.to_csv(CROP_REPORT_CSV, index=False)
print(f"Crop output: {CROP_OUTPUT_DIR.resolve()}")
print(f"Report CSV: {CROP_REPORT_CSV.resolve()}")

saved_df = report_df[report_df["status"] == "saved"].copy()
if saved_df.empty:
    print("No saved crops to display.")
else:
    preview_paths = [Path(p) for p in saved_df["out_path"].head(6).tolist()]

    n = len(preview_paths)
    cols = 3
    rows_n = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 4 * rows_n))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, p in zip(axes, preview_paths):
        img = Image.open(p)
        ax.imshow(img)
        ax.set_title(p.name)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
